# Expfit notes 3: Confidence intervals

We'd like to set confidence intervals, particularly on obtained time constants.

## Relation to maximum likelihood estimation

So far, we optimised the mean squared error (MSE) to find a best fit

\begin{align}
\hat{E} = E(\hat{\theta}) = \frac{1}{n}\sum(v_i - f(\hat{\theta})_i)^2
\end{align}

where $\theta$ is a parameter vector of size $m$, $v_i$ are the data, and $f(\theta)$ is the model (a sum of exponentials).

Some nice results from statistics can be used to set confidence intervals, or to decide whether to add more exponentials.
To do this, we need to reformulate as a statistical problem by replacing the MSE with a log likelihood.

First, we'll assume the model $f$ is a sum of exponentials plus a noise variable distributed as $N(0, \sigma^2)$ for some unknown $\sigma$ _even though this may not be true of our data_.

Then, the likelihood of observing $v$ is
\begin{align}
L(\theta, \sigma | v) \equiv p(v | \theta, \sigma)
 &= \prod^n_{i=1} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( - \frac{(v_i - f(\theta)_i)^2}{2\sigma^2} \right)
\end{align}
leading to a log-likelihood of
\begin{align}
\ell(\theta, \sigma | v) \equiv \log L(\theta, \sigma | v)
 &= -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{1}{2\sigma^2} \sum^n_{i=1} (v_i - f(\theta)_i)^2 \\
 &= -\frac{n}{2}\log(2\pi) - n\log(\sigma) - \frac{n}{2\sigma^2} E(\theta)
\end{align}

If we assume that $n$ and $\sigma$ are fixed, then the first two terms are constant, and so the $\theta$ that minimises the MSE also maximises the likelihood.

### Approximating $\sigma^2 \approx \hat{E}$

The (assumed Gaussian) noise level $\sigma$ can be estimated from a flat bit of signal.

We can also subtract a straight line from a straight bit of signal, or even a single exponential from a nicely exponential bit of data.
In other words, if our model fits well, we can estimate $\sigma^2$  as the sample variance of the residuals.
This is exactly the same equation as the MSE:
\begin{align}
\sigma^2 \approx \frac{1}{n}\sum(v_i - f(\hat{\theta})_i)^2 \equiv \hat{E}
\end{align}
so that we can use
\begin{align}
\ell(\theta, \sigma | v) \equiv \log L(\theta, \sigma | v) \approx
  -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\hat{E}) - \frac{n}{2\hat{E}}E(\theta)
\end{align}

### The Hessian

For the "profile log-likelihood" method described below, we need the second derivative of $\mathcal{l}$.

Because $n$ and $\sigma^2$ are constant (and depend on the data, not the model), the first two terms disappear and we obtain
\begin{align}
\frac{\partial^2 \ell(\theta)}{\partial\theta_i\partial\theta_j} 
  = - \frac{n}{2\sigma^2} H_{ij}(\theta)
  \approx - \frac{n}{2\hat{E}} H_{ij}(\theta)
\end{align}

## Profile log-likelihood confidence intervals

After finding an optimal fit $\hat{\theta}$, we can find a confidence interval (CI) around the i-the entry in $\hat{theta}$ using a profile log likelihood method.
This proceeds as follows:

1. Fix the parameter of interest at a distance $d > 0$ from its original value
2. Formulate a new optimisation problem over the remaining $m - 1$ parameters, yielding a "test" vector $\theta_t$ and associated $E_t = E(\theta_t)$
3. While $E_t$ is below some acceptable level, increase $d$ and repeat.
4. When an upper bound for $d$ has been found ($E_t$ has exceed the acceptable level), start a bisection search to find the value $d$ for which $E_t$ is exactly at the cut-off point

This gives an upper bound.
The procedure is repeated with $d < 0$ to find the lower bound.

This requires repeated reoptimisation, but for these problems it should still be very fast.

For the "acceptable level" of error, we can intuitively define some cut-off
\begin{align}
E_\text{max} = (1 + c) \hat{E}
\end{align}
where $c$ is a small positive number.

But we can also arrive at this using the statistical approach. In particular, we can compare the likelihood of our test value $\theta_t$ with the likelihood of the optimum $\hat{\theta}$ using a [likelihood ratio test](https://en.wikipedia.org/wiki/Likelihood-ratio_test):

\begin{align}
-2 \log \frac{L(\theta_t)}{L(\hat{\theta})} = -2 (\mathcal{l}(\theta_t) - \mathcal{l}(\hat{\theta})) < \chi^2_{1,1-\alpha}
\end{align}

where $\chi_{1,1-\alpha}$ is the [percent point or "quantile" function](https://en.wikipedia.org/wiki/Quantile_function) of a [Chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) with $1$ degree of freedom and $1-\alpha$ is the level of confidence (see [Wilks theorem](https://en.wikipedia.org/wiki/Wilks%27_theorem)).
For example, if we want a CI that contains the true solution with 95% probability, we use $\chi_{1,0.95} \approx 3.841$.
The "one degree of freedom" here refers to the different parameter vector sizes: $m$ for the original $\hat{\theta}$ and $m - 1$ for the tested value $\theta_t$.
(So if we had fixed two parameters, an extension to this method, we would use the 2 degrees of freedom statistic.)

Values of $\chi^2$ can be found in tables, or obtained from Scipy:

In [1]:
import scipy
print(scipy.stats.chi2.ppf(0.99, 1))
print(scipy.stats.chi2.ppf(0.95, 1))
print(scipy.stats.chi2.ppf(0.90, 1))

6.6348966010212145
3.841458820694124
2.705543454095404


_Note that the "level of confidence" refers to the chances of the true solution being within our interval, not the certainty of our optimum! So increased confidence (the 0.99 value) means a much wider interval._

Using the relationship between $\mathcal{l}$ and $E$ given above, we can rewrite this inequality to find a cut-off in terms of our chosen $\chi^2$.
Here, we take advantage of the fact that $n$ and $\sigma$ are the same for $\mathcal{l}(\theta_t)$ and $\mathcal{l}(\hat{\theta})$ so that only the final terms of the likelihoods remain, for
\begin{align}
-2 (\mathcal{l}(\theta_t) - \mathcal{l}(\hat{\theta})) = \frac{n}{\sigma^2} (E(\theta_t) - \hat{E}) \approx n \left(\frac{E_t(\theta)}{\hat{E}} - 1\right)
\end{align}
for
\begin{align}
n \left(\frac{E_t(\theta)}{\hat{E}} - 1\right) < \chi^2_{1,1-\alpha}
\end{align}
which means our confidence interval includes all values for which
\begin{align}
E_t(\theta) < (1 + \chi^2_{1,1-\alpha} / n) \hat{E}
\end{align}

Note the appearance of $n$: more data gives tigther bounds.

## Fisher information confidence intervals

A cheaper, but sometimes less accurate method uses [Fisher information](https://en.wikipedia.org/wiki/Fisher_information).
This starts from a property of maximum likelihood estimation (MLE) called _asymptotic normality_, which states that near the true solution $\theta_\text{true}$, the _maximum likelihood estimate_ $\hat{\theta}$ is Normally distributed:
\begin{align}
\hat{\theta} \sim N(\theta_\text{true}, I^{-1}(\theta_\text{true}))
\end{align}
Here we treat $\hat{\theta}$ as a random variable because it's based on randomly sampled data - resampling the data $v$ with the same $\theta_\text{true}$ would provide a different estimate $\hat{\theta}$.
(We'll assume the optimiser is perfect and deterministic.)

The quantity $I(\theta)$ is called the Fisher information matrix, and is defined as
\begin{align}
I_{ij}(\theta) = \mathbb{E} \left( -\frac{\partial^2\mathcal{l}(\theta|v)}{\partial \theta_i \partial \theta_j} \right)
\end{align}
where $\mathbb{E}()$ is the expected value.
We don't know this, but can approximate as
\begin{align}
I_{ij}(\theta) \approx J_{ij}(\hat{\theta}) 
    = \left. -\frac{\partial^2\mathcal{l}(\theta|v)}{\partial \theta_i \partial \theta_j} \right\vert_{\theta = \hat{\theta}}
    = \frac{n}{2\sigma^2} H_{ij}(\hat{\theta})
    \approx \frac{n}{2\hat{E}} H_{ij}(\hat{\theta})
\end{align}
leading to an approximate covariance matrix
\begin{align}
\Sigma = J^{-1}(\hat{\theta}) = \frac{2\hat{E}}{n} H^{-1}(\hat{\theta})
\end{align}

To find the confidence interval, we approximate the distribution of the obtained optimum as
\begin{align}
\hat{\theta} \sim N(\theta_\text{true}, \Sigma(\hat{\theta}))
\end{align}
then rewrite for a single parameter (using $\hat{\Sigma} = \Sigma(\hat{\theta})$)
\begin{align}
\hat{\theta}_i \sim N(\theta_{\text{true}_i}, \hat{\Sigma}_{ii})
\end{align}
and standardise
\begin{align}
\frac{\hat{\theta}_i - \theta_{\text{true}_i}}{\sqrt{\hat{\Sigma}_{ii}}} \sim N(0, 1)
\end{align}
Because this is a symmetric distribution, using the 95% percentile point function of a Normal distribution gives a 90% confidence bound:
\begin{align}
CI_{90} = \hat{\theta}_i \pm 1.645 \sqrt{\hat{\Sigma}_{ii}}
\end{align}

As before, we can use Scipy to find these percentile points:

In [2]:
print(scipy.stats.norm.ppf(0.975))  # 95%
print(scipy.stats.norm.ppf(0.95))   # 90%

1.959963984540054
1.6448536269514722


### Quadratic approximation of the MSE

We can construct a quadratic approximation to the MSE near the optimum, as
\begin{align}
E(\theta_i) \approx \hat{E} + \frac{1}{2} H_{ii} (\theta_i - \hat{\theta}_i)^2
\end{align}
This is similar to varying a similar parameter, but keeping all others fixed, and so will be very steep.

We can use more information from the $H$ to construct a quadratic approximation related to the Fisher information (and resulting CI) as
\begin{align}
E(\theta_i) \approx \hat{E} + \frac{1}{2} \frac{1}{H^{-1}_{ii}} (\theta_i - \hat{\theta}_i)^2
\end{align}
where the hessian is first inverted, and then one-over the diagonal entries is used.

## Model selection

To compare fits with different numbers of exponentials, we can use the [Akaike information criterion](https://en.wikipedia.org/wiki/Akaike_information_criterion):
\begin{align}
\mathcal{A} 
= 2m - 2 \ell(\theta) 
= 2m + n\log(2\pi) + 2n\log(\sigma) + \frac{n}{\sigma^2} \hat{E}
\end{align}
where $m$ is the number of parameters in the model (1 for the offset, and then 2 per exponential).

A lower AIC indicates a better model.
So we select the model with the lowest 
\begin{align}
\mathcal{A}^* = 2m + \frac{n}{\sigma^2} \hat{E}_m
\end{align}
where $\hat{E}_m$ is the best solution found for the $m$-parameter model.

Note that we have to use the same $\sigma^2$ approximation for both models.

### Ratio likelihood test

**TODO: Instead, use a likelihood ratio test https://en.wikipedia.org/wiki/Likelihood-ratio_test with chi-squared for the different no of parameters (2 per added expo)**